# Xarray Fundamentals

## Intro to Xarray 

[Xarray for multidimensional gridded data](https://earth-env-data-science.github.io/lectures/xarray/xarray_intro.html).

## Xarray data structures

Like Pandas, xarray has two fundamental data structures:
* a `DataArray`, which holds a single multi-dimensional variable and its coordinates
* a `Dataset`, which holds multiple variables that potentially share the same coordinates

### DataArray

A `DataArray` has four essential attributes:
* `values`: a `numpy.ndarray` holding the array’s values
* `dims`: dimension names for each axis (e.g., `('x', 'y', 'z')`)
* `coords`: a dict-like container of arrays (coordinates) that label each point (e.g., 1-dimensional arrays of numbers, datetime objects or strings)
* `attrs`: an `OrderedDict` to hold arbitrary metadata (attributes)

Let's start by constructing some DataArrays manually 

In [ ]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (8,5)

A simple DataArray without dimensions or coordinates isn't much use.

In [ ]:
da = xr.DataArray([9, 0, 2, 1, 0])
da

We can add a dimension name...

In [ ]:
da = xr.DataArray([9, 0, 2, 1, 0], dims=['x'])
da

But things get most interesting when we add a coordinate:

In [ ]:
da = xr.DataArray([9, 0, 2, 1, 0],
                  dims=['x'],
                  coords={'x': [10, 20, 30, 40, 50]})
da

This coordinate has been used to create an _index_, which works very similar to a Pandas index.
In fact, under the hood, Xarray just reuses Pandas indexes.

In [ ]:
da.indexes

Xarray has built-in plotting, like pandas.

In [ ]:
da.plot(marker='o')

### Multidimensional DataArray

If we are just dealing with 1D data, Pandas and Xarray have very similar capabilities. Xarray's real potential comes with multidimensional data.

Let's go back to the multidimensional ARGO data we loaded in the numpy lesson.

In [ ]:
import pooch
url = "https://dsrs.atmos.umd.edu/DATA/445Files/Argov2.zip"
files = pooch.retrieve(url, processor=pooch.Unzip(),known_hash = '4565d7dfc16947e4226275e553da4726a30cee9cbd56d08ee28e670907711aba')
files.sort()
files

We will manually load each of these variables into a numpy array.
If this seems repetitive and inefficient, that's the point!
NumPy itself is not meant for managing groups of inter-related arrays.
That's what Xarray is for!

In [ ]:
P = np.load(files[4])
S = np.load(files[5])
T = np.load(files[6])
date = np.load(files[0])
lat = np.load(files[1])
levels = np.load(files[2])
lon = np.load(files[3])

In [ ]:
S.shape

Let's organize the data and coordinates of the salinity variable into a DataArray.

In [ ]:
da_salinity = xr.DataArray(S.T, dims=['level', 'date'],
                           coords={'level': levels,
                                   'date': date},)
da_salinity

In [ ]:
da_salinity.plot(yincrease=False)

Attributes can be used to store metadata. What metadata should you store? The [CF Conventions](http://cfconventions.org/Data/cf-conventions/cf-conventions-1.7/cf-conventions.html#_description_of_the_data) are a great resource for thinking about climate metadata. Below we define two of the required CF-conventions attributes.

In [ ]:
da_salinity.attrs['units'] = 'PSU'
da_salinity.attrs['standard_name'] = 'sea_water_salinity'
da_salinity

Now if we plot the data again, the name and units are automatically attached to the figure.

In [ ]:
da_salinity.plot(yincrease=False)

### Datasets

A Dataset holds many DataArrays which potentially can share coordinates. In analogy to pandas:

    pandas.Series : pandas.Dataframe :: xarray.DataArray : xarray.Dataset
    
Constructing Datasets manually is a bit more involved in terms of syntax. The Dataset constructor takes three arguments:

* `data_vars` should be a dictionary with each key as the name of the variable and each value as one of:
  * A `DataArray` or Variable
  * A tuple of the form `(dims, data[, attrs])`, which is converted into arguments for Variable
  * A pandas object, which is converted into a `DataArray`
  * A 1D array or list, which is interpreted as values for a one dimensional coordinate variable along the same dimension as it’s name
* `coords` should be a dictionary of the same form as data_vars.
* `attrs` should be a dictionary.

Let's put together a Dataset with temperature, salinity and pressure all together

In [ ]:
argo = xr.Dataset(
    data_vars={
        'salinity':    (('level', 'date'), S.T),
        'temperature': (('level', 'date'), T.T),
        'pressure':    (('level', 'date'), P.T)
    },
    coords={
        'level': levels,
        'date': date
    }
)
argo

What about lon and lat? We forgot them in the creation process, but we can add them after the fact.

We want lon to have dimension `date`:

In [ ]:
argo.coords['lon'] = ('date', lon)
argo.coords['lat'] = ('date', lat)
argo

## Selecting Data (Indexing)

We can always use regular numpy indexing and slicing on DataArrays

In [ ]:
argo.salinity[2]

In [ ]:
argo.salinity[2].plot()

In [ ]:
argo.salinity[:, 10]

In [ ]:
argo.salinity[:, 10].plot()

However, it is often much more powerful to use xarray's `.sel()` method to use label-based indexing.

In [ ]:
argo.salinity.sel(level=2)

In [ ]:
argo.salinity.sel(level=2).plot()

In [ ]:
argo.salinity.sel(date='2023-09-07')

In [ ]:
argo.salinity.sel(date='2023-09-07').plot(y='level', yincrease=False)

`.sel()` also supports slicing. Unfortunately we have to use a somewhat awkward syntax, but it still works.

In [ ]:
argo.salinity.sel(date=slice('2023-09-07', '2023-12-01'))

In [ ]:
argo.salinity.sel(date=slice('2023-09-07', '2023-12-01')).plot(yincrease=False)

`.sel()` also works on the whole Dataset

In [ ]:
argo.sel(date='2023-09-07')

### Practice
1) Select and plot temperature at two different levels using both indexing and .sel
2) Plot temperature profile for 'date' in the dataset.
3) Plot temperature over a range of days.  
4) Plot trajectory of argo float using scatter

## Computation

Xarray dataarrays and datasets work seamlessly with arithmetic operators and numpy array functions.

In [ ]:
temp_kelvin = argo.temperature + 273.15
temp_kelvin.plot(yincrease=False)

We can also combine multiple xarray datasets in arithemtic operations

In [ ]:
g = 9.8
buoyancy = g * (2e-4 * argo.temperature - 7e-4 * (argo.salinity-35))
buoyancy.plot(yincrease=False)

## Broadcasting, Aligment, and Combining Data

### Broadcasting

Broadcasting arrays in numpy is a nightmare. It is much easier when the data axes are labeled!

This is a useless calculation, but it illustrates how perfoming an operation on arrays with differenty coordinates will result in automatic broadcasting

In [ ]:
level_times_lat = argo.level * argo.lat
(argo.level.shape, argo.lat.shape, level_times_lat.shape)

In [ ]:
level_times_lat.plot()

### Alignment

If you try to perform operations on DataArrays that share a dimension name, Xarray will try to _align_ them first.
This works nearly identically to Pandas, except that there can be multiple dimensions to align over.

To see how alignment works, we will create some subsets of our original data.

In [ ]:
sa_surf = argo.salinity.isel(level=slice(0, 200))
sa_mid = argo.salinity.isel(level=slice(100, 300))

By default, when we combine multiple arrays in mathematical operations, Xarray performs an "inner join".

In [ ]:
(sa_surf * sa_mid).level
# sa_surf.level
# sa_mid.level

We can override this behavior by manually aligning the data

In [ ]:
sa_surf_outer, sa_mid_outer = xr.align(sa_surf, sa_mid, join='outer')
sa_surf_outer.level
# sa_mid_outer.level

As we can see, missing data (NaNs) have been filled in where the array was extended.

In [ ]:
sa_surf_outer.plot(yincrease=False)

We can also use `join='right'` and `join='left'`.

### Combing Data: Concat and Merge

The ability to combine many smaller arrays into a single big Dataset is one of the main advantages of Xarray.
To take advantage of this, we need to learn two operations that help us combine data:
- `xr.concat`: to concatenate multiple arrays into one bigger array along their dimensions
- `xr.merge`: to combine multiple different arrays into a dataset

First let's look at concat. Let's re-combine the subsetted data from the previous step.

In [ ]:
sa_surf_mid = xr.concat([sa_surf, sa_mid], dim='level')
sa_surf_mid

```{warning}
Xarray won't check the values of the coordinates before `concat`. It will just stick everything together into a new array.
```

In this case, we had overlapping data. We can see this by looking at the `level` coordinate.

In [ ]:
sa_surf_mid.level

In [ ]:
plt.plot(sa_surf_mid.level.values, marker='o')

We can also concat data along a _new_ dimension, e.g.

In [ ]:
sa_concat_new = xr.concat([sa_surf, sa_mid], dim='newdim')
sa_concat_new

# sa_concat_new[0,:,:].plot()

Note that the data were aligned using an _outer_ join along the non-concat dimensions.

Instead of specifying a new dimension name, we can pass a new Pandas index object explicitly to `concat`.
This will create a new dimension coordinate and corresponding index.

We can merge both DataArrays and Datasets.

In [ ]:
xr.merge([argo.salinity, argo.temperature])

If the data are not aligned, they will be aligned before merge.
We can specify the join options in `merge`.

In [ ]:
xr.merge([
    argo.salinity.sel(level=slice(0, 300)),
    argo.temperature.sel(level=slice(300, None)) # join='outer' as default
])

In [ ]:
xr.merge([
    argo.salinity.sel(level=slice(0, 300)),
    argo.temperature.sel(level=slice(300, None))
], join='left')

## Reductions

Just like in numpy, we can reduce xarray DataArrays along any number of axes:

In [ ]:
argo.temperature.mean(axis=0)

In [ ]:
argo.temperature.mean(axis=1)

However, rather than performing reductions on axes (as in numpy), we can perform them on dimensions. This turns out to be a huge convenience

In [ ]:
argo_mean = argo.mean(dim='date')
argo_mean

In [ ]:
argo_mean.salinity.plot(y='level', yincrease=False)

In [ ]:
argo_std = argo.std(dim='date')
argo_std.salinity.plot(y='level', yincrease=False)

### Weighted Reductions

Sometimes we want to perform a reduction (e.g. a mean) where we assign different weight factors to each point in the array.
Xarray supports this via [weighted array reductions](http://xarray.pydata.org/en/stable/user-guide/computation.html#weighted-array-reductions).

As a toy example, imagine we want to weight values in the upper ocean more than the lower ocean.
We could imagine creating a weight array exponentially proportional to pressure as follows:

In [ ]:
mean_pressure = argo.pressure.mean(dim='date')
p0 = 250  # dbat
weights = np.exp(-mean_pressure / p0)
weights.plot()

The weighted mean over the `level` dimensions is calculated as follows:

In [ ]:
temp_weighted_mean = argo.temperature.weighted(weights).mean('level')

Comparing to the unweighted mean, we see the difference:

In [ ]:
temp_weighted_mean.plot(label='weighted')
argo.temperature.mean(dim='level').plot(label='unweighted')
plt.legend()

## Loading Data from netCDF Files

NetCDF (Network Common Data Format) is the most widely used format for distributing geoscience data. NetCDF is maintained by the [Unidata](https://www.unidata.ucar.edu/) organization.

Below we quote from the [NetCDF website](https://www.unidata.ucar.edu/software/netcdf/docs/faq.html#whatisit):

>NetCDF (network Common Data Form) is a set of interfaces for array-oriented data access and a freely distributed collection of data access libraries for C, Fortran, C++, Java, and other languages. The netCDF libraries support a machine-independent format for representing scientific data. Together, the interfaces, libraries, and format support the creation, access, and sharing of scientific data.
>
>NetCDF data is:
>
> - Self-Describing. A netCDF file includes information about the data it contains.
> - Portable. A netCDF file can be accessed by computers with different ways of storing integers, characters, and floating-point numbers.
> - Scalable. A small subset of a large dataset may be accessed efficiently.
> - Appendable. Data may be appended to a properly structured netCDF file without copying the dataset or redefining its structure.
> - Sharable. One writer and multiple readers may simultaneously access the same netCDF file.
> - Archivable. Access to all earlier forms of netCDF data will be supported by current and future versions of the software.

We can use [NetCDF Operators (NCO)](https://nco.sourceforge.net/) to process and analyze netCDF files. Useful NCO commands include ncdump, ncks (Kitchen Sink), ncrcat (conCATenator), ncwa (Weighted Averager).

The metadata in netCDF files are recommended to follow the [Climate and Forecasting (CF) conventions](https://cfconventions.org/cf-conventions/cf-conventions.html).

Xarray was designed to make reading netCDF files in python as easy, powerful, and flexible as possible. (See [xarray netCDF docs](http://xarray.pydata.org/en/latest/io.html#netcdf) for more details.)

Below we download and load some the NASA [GISSTemp](https://data.giss.nasa.gov/gistemp/) global temperature anomaly dataset.

In [ ]:
## doesn't work anymore... Forbidden for url: https://data.giss.nasa.gov/pub/gistemp/gistemp1200_GHCNv4_ERSSTv5.nc.gz
# gistemp_file = pooch.retrieve(
#      'https://data.giss.nasa.gov/pub/gistemp/gistemp1200_GHCNv4_ERSSTv5.nc.gz',
#      known_hash='2a703c720302c682f1662181d329c9f22f9f10e1539dc2d6082160a469165009',
#      processor=pooch.Decompress(),
# )

In [ ]:
## instead a copy of the data is uploaded to zenodo 
import pooch
doi = "doi:10.5281/zenodo.13963679"
fname = "gistemp1200_GHCNv4_ERSSTv5.nc.gz"
gistemp_file = pooch.retrieve(
    url=f"{doi}/{fname}",
    known_hash="md5:f34b27a0c3d2f052ae9a24621aa86aac",
    processor=pooch.Decompress(),
)
gistemp_file

In [ ]:
ds = xr.open_dataset(gistemp_file)
ds

In [ ]:
ds.tempanomaly.isel(time=-1).plot()

In [ ]:
ds.tempanomaly.mean(dim=('lon', 'lat')).plot()

But wait! This is not exactly right. We need to weight the spatial mean by the area of each grid cell.

In [ ]:
weights = np.cos(np.deg2rad(ds.lat))
ds.tempanomaly.mean(dim=('lon', 'lat')).plot(label='unweighted')
ds.tempanomaly.weighted(weights).mean(dim=('lon', 'lat')).plot(label='weighted')
plt.legend()

### Practice
1) Plot the surface temperature anomaly along the prime meridian for 1980
2) Calculate the weighted-average temperature anomaly over the northern hemisphere versus southern hemisphere
3) Evaluate the zonally averaged (east-west) trend in temperature and plot vs latitude.